# Stage datasets and weights into DBFS

Downloads everything the experiments read into `/dbfs`, so it survives cluster restarts and no training run ever waits on a download:

| what | where | read by |
|---|---|---|
| CIFAR-10 | `/dbfs/cache` | every CV run (`training_utils` sets `cache_dir=/dbfs/cache` when `databricks_env` is on) |
| SST-2 + tokenizers | `/dbfs/cache` | every LLM run |
| ImageNet weights (resnet34/50, vgg11/19) | `/dbfs/research/bacp/` | the registry paths `model_factory.MODELS` points at |

Safe to re-run: everything checks before downloading. Off Databricks the datasets stage into the local `./cache` equivalents, but the ImageNet weights always go to the registry paths under `/dbfs/...` (created as a plain directory on a non-Databricks machine) -- that is where `model_factory.MODELS` loads from on every platform.

In [ ]:
import sys, pathlib

# Find nb_common.py whether the kernel started in this folder or at the repo root.
here = pathlib.Path.cwd()
for cand in [here, *here.parents]:
    if (cand / 'nb_common.py').exists():
        sys.path.insert(0, str(cand)); break
    if (cand / 'project' / 'test_notebooks' / 'nb_common.py').exists():
        sys.path.insert(0, str(cand / 'project' / 'test_notebooks')); break
else:
    raise RuntimeError('cannot find nb_common.py -- start the kernel inside the repo')

import nb_common as nb
info = nb.setup()

## CIFAR-10

In [ ]:
import os
CACHE = '/dbfs/cache' if info['databricks'] else str(nb.REPO / 'project' / 'scripts' / 'cache')
os.makedirs(CACHE, exist_ok=True)
print('cache dir:', CACHE)

from torchvision.datasets import CIFAR10
for train in (True, False):
    CIFAR10(CACHE, train=train, download=True)
print('CIFAR-10 ready')

## SST-2 + tokenizers

In [ ]:
from datasets import load_dataset
ds = load_dataset('glue', 'sst2', cache_dir=CACHE)
print({split: len(ds[split]) for split in ds})

from transformers import AutoTokenizer
for name in ('distilbert-base-uncased', 'roberta-base'):
    AutoTokenizer.from_pretrained(name, cache_dir=CACHE)
    print('tokenizer cached:', name)

## ImageNet weights at the registry paths

These are the exact paths `model_factory.MODELS` loads from; the preflight in every notebook refuses to train unless they resolve.

In [ ]:
for model in ('resnet34', 'resnet50', 'vgg11', 'vgg19'):
    nb.fetch_imagenet_weights(model)

## Verify

In [ ]:
import os
print('--- /dbfs contents this notebook manages ---' if info['databricks'] else '--- local cache ---')
for base in (CACHE, '/dbfs/research/bacp'):
    if not os.path.isdir(base):
        print(f'{base}: MISSING'); continue
    total = 0
    for dirpath, _, files in os.walk(base):
        total += sum(os.path.getsize(os.path.join(dirpath, f)) for f in files)
    print(f'{base}: {total/1e6:,.0f} MB')